In [1]:
import csv
import os
import pickle

import numpy as np
import pandas as pd
pd.options.display.max_columns = 100

In [2]:
# Chemin (absolu ou relatif) du jeu de données, après avoir décompressé le fichier correspondant
# A MODIFIER EN FONCTION DE LA LOCALISATION DE CE DOSSIER SUR VOTRE MACHINE
path = os.path.join("datasets", "tennis")

# Lecture des données

Il existe plusieurs façons de lire des données tabulaires en Python :

* On peut le faire en Python pur, sans aucune bibliothèque, avec la fonction native [`open()`](https://docs.python.org/fr/3.14/library/functions.html#open).
* On peut utiliser une bibliothèque parmi la bibliothèque standard telle que [`csv`](https://docs.python.org/fr/3/library/csv.html) ou [`json`](https://docs.python.org/fr/3/library/json.html) suivant le format des fichiers.
* On peut utiliser une bibliothèque tierce, en fonction des types de données et des préférences personnelles, telle que [`pandas`](https://pandas.pydata.org), [`polars`](https://pola.rs), [`numpy`](https://numpy.org/doc/stable/), etc.

Vous trouverez ci-dessous des exemples pour chacune de ces méthodes :

In [3]:
list_player = []
with open(os.path.join(path, "wta_players_2024.csv")) as file:
    for line in file.readlines():
        list_player.append(line.split(","))
        
list_player[:3]

[['player_id', 'name_first', 'name_last', 'hand', 'dob', 'ioc', 'height\n'],
 ['200748', 'Venus', 'Williams', 'R', '19800617.0', 'USA', '185.0\n'],
 ['201427', 'Alize', 'Cornet', 'R', '19900122.0', 'FRA', '173.0\n']]

In [4]:
list_csv_player = list(csv.reader(open(os.path.join(path, "wta_players_2024.csv"))))
list_csv_player[:3]

[['player_id', 'name_first', 'name_last', 'hand', 'dob', 'ioc', 'height'],
 ['200748', 'Venus', 'Williams', 'R', '19800617.0', 'USA', '185.0'],
 ['201427', 'Alize', 'Cornet', 'R', '19900122.0', 'FRA', '173.0']]

In [5]:
df_player = pd.read_csv(os.path.join(path, "wta_players_2024.csv"))
df_player.head(2)

,player_id,name_first,name_last,hand,dob,ioc,height
0,200748,Venus,Williams,R,19800617.0,USA,185.0
1,201427,Alize,Cornet,R,19900122.0,FRA,173.0


In [6]:
# Vérification de la présence d'éventuelles valeurs manquantes pour chaque colonne
df_player.isna().any(axis=0)

player_id     False
name_first    False
name_last     False
hand          False
dob            True
ioc           False
height         True
dtype: bool

# Création d'une classe pour modéliser une joueuse

In [7]:
import datetime


class Player:
    def __init__(
        self,
        lastname: str,
        firstname: str,
        birthdate: datetime.date | None,
        country: str,
        hand: str,
        height: int | None
    ) -> None:
        self.lastname = lastname
        self.firstname = firstname
        self.birthdate = birthdate
        self.country = country
        self.hand = hand
        self.height = height
        self.statistiques = {}
            
    def ajouter_statistiques(self, annee: str, statistiques: dict) -> None:
        self.statistiques[annee] = statistiques

# Lecture du fichier des matchs et calcul de statistiques

In [8]:
df_match = pd.read_csv(os.path.join(path, "wta_matches_2024.csv"))
df_match.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,loser_id,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced
0,2024-9900,United Cup,Hard,18,I,20240101,299,216347,201493,6-3 6-0,3,F,70.0,1.0,3.0,59.0,43.0,31.0,9.0,8.0,5.0,5.0,0.0,0.0,42.0,27.0,13.0,7.0,7.0,4.0,8.0
1,2024-9900,United Cup,Hard,18,I,20240101,297,216347,201614,4-6 6-1 6-1,3,SF,106.0,2.0,4.0,76.0,54.0,41.0,10.0,12.0,3.0,4.0,2.0,1.0,66.0,47.0,28.0,7.0,12.0,3.0,8.0
2,2024-9900,United Cup,Hard,18,I,20240101,295,201493,201548,4-6 6-2 7-6(7),3,SF,154.0,1.0,1.0,99.0,78.0,45.0,8.0,15.0,6.0,13.0,3.0,4.0,116.0,82.0,47.0,10.0,15.0,11.0,19.0
3,2024-9900,United Cup,Hard,18,I,20240101,293,216347,221012,6-2 6-3,3,QF,94.0,0.0,1.0,55.0,32.0,21.0,14.0,9.0,1.0,3.0,2.0,9.0,57.0,27.0,18.0,6.0,8.0,6.0,11.0
4,2024-9900,United Cup,Hard,18,I,20240101,291,201614,214935,6-2 6-7(6) 7-6(5),3,QF,153.0,15.0,4.0,101.0,62.0,46.0,28.0,16.0,1.0,2.0,4.0,6.0,111.0,72.0,51.0,22.0,16.0,3.0,6.0


In [9]:
def calculer_nombre_tournois_gagnes() -> pd.Series:
    # Récupération de toutes les joueuses
    players = pd.concat([df_match["winner_id"], df_match["loser_id"]]).unique()

    # Initialisation du résultat renvoyé : 0 tournoi gagné pour chaque joueuse
    res = pd.Series(data=0, index=players, name="n_tournaments_won")

    # Calcul du nombre de tournois gagnés pour les joueuses ayant gagné au moins 1 tournoi
    winners = (
        df_match
        .loc[df_match["round"] == "F", ["winner_id", "tourney_id"]]
        .groupby("winner_id")["tourney_id"].nunique()
    )

    # Mise à jour du résultat renvoyé
    res.loc[winners.index] = winners

    # Renvoi du résultat
    return res

In [10]:
def calculer_taux_victoires() -> pd.Series:
    # Récupérer toutes les joueuses
    players = pd.concat([df_match["winner_id"], df_match["loser_id"]]).unique()

    # Initialisation du nombre de vcitoires et de défaites par joueuse à 0
    wins = pd.Series(data=0, index=players)
    losses = pd.Series(data=0, index=players)
    
    # Calcul du nombre de victoires et de défaites par joueuse
    wins_actual = df_match["winner_id"].value_counts()
    losses_actual = df_match["loser_id"].value_counts()
    
    # Mise à jour des objets initiaux
    wins.loc[wins_actual.index] = wins_actual
    losses.loc[losses_actual.index] = losses_actual
    
    # Renvoi du résultat
    res = wins / (wins + losses)
    res.name = "winning_ratio"
    return res

In [11]:
def calculer_meilleur_resultat_grand_chelem() -> pd.Series:
    # Initialisation du résultat renvoyé
    players = pd.concat([df_match["winner_id"], df_match["loser_id"]]).unique()
    res = pd.Series(data=None, index=players, dtype=str, name="best_grand_chelem_result")
    
    # Sélection des matchs en grand chelem
    df_match_g = df_match[df_match["tourney_level"] == "G"].copy()
    
    # Correspondances entre les tours et leurs niveaux
    mapping_round_int = {"R128": 0, "R64": 1, "R32": 2, "R16": 3, "QF": 4, "SF": 5, "F": 6}
    mapping_int_round = {value: key for key, value in mapping_round_int.items()}
    
    # Création d'une variable temporaire pour ordonner les tours
    df_match_g["round_int"] = df_match_g["round"].map(mapping_round_int)
    
    # Récupération du meilleur résultat pour les joueuses ayant participé à un grand chelem
    # (hors victoire d'un tournoi)
    best_results = (
        df_match_g[df_match_g["tourney_level"] == "G"]
        .groupby("loser_id")["round_int"].max()
        .map(mapping_int_round)
    )

    # Récupération des vainqueuses des tournois du grand chelem
    winners = df_match_g.loc[df_match_g["round"] == "F", "winner_id"].to_numpy()

    # Mise à jour du résultat renvoyé
    res.loc[best_results.index] = best_results
    res.loc[winners] = "W"

    # Renvoi du résultat
    return res

# Création de tous les objets pour les joueuses

In [12]:
def creer_objets_joueuses() -> dict[int, Player]:
    df_player = pd.read_csv(os.path.join(path, "wta_players_2024.csv"))
    df_match = pd.read_csv(os.path.join(path, "wta_matches_2024.csv"))
    
    df_statistics = pd.concat([
        calculer_nombre_tournois_gagnes(),
        calculer_taux_victoires(),
        calculer_meilleur_resultat_grand_chelem(),
    ], axis=1)
    
    mapping_hand = {"L": "gauche", "R": "droite", "U": "inconnue"}
    
    res = {}
    for record in df_player.to_dict("records"):
        
        # Date de naissance
        if not np.isnan(record["dob"]):
            birthdate = datetime.datetime.strptime(f"{record["dob"]:.0f}", "%Y%m%d")
            birthdate = datetime.date(birthdate.year, birthdate.month, birthdate.day)
        else:
            birthdate = None
        
        # Taille
        height = int(record["height"]) if not np.isnan(record["height"]) else None

        res[record["player_id"]] = Player(
            lastname=record["name_first"],
            firstname=record["name_last"],
            birthdate=birthdate,
            country=record["ioc"],
            hand=mapping_hand[record["hand"]],
            height=height,
        )
        
        res[record["player_id"]]

    dict_statistics = df_statistics.to_dict("index")
    for key, value in dict_statistics.items():
        res[key].ajouter_statistiques(2024, value)

    return res

# Sauvegarde des objets

Il peut être pertinent de sauvegarder tous les objets que vous aurez créés dans des fichiers afin de ne pas à avoir à les créer à chaque fois que vous lancerez votre application.
Il existe plusieurs manières de sauvegarder des objets Python sur un disque dur :

* La bibliothèque [`pickle`](https://docs.python.org/fr/3.14/library/pickle.html) fait partie de la bibliothèque standard : [`pickle.dump()`](https://docs.python.org/fr/3.14/library/pickle.html#pickle.dump) permet de sauvegarder un objet Python dans un fichier et [`pickle.load()`](https://docs.python.org/fr/3.14/library/pickle.html#pickle.load) fait l'opération inverse.
    
* La bibliothèque tierce [`numpy`](https://numpy.org/doc/stable/) permet de sauvegarder et lire des tableaux NumPy grâce aux fonctions [`numpy.save()`](https://numpy.org/doc/stable/reference/generated/numpy.save.html) et [`numpy.load()`](https://numpy.org/doc/stable/reference/generated/numpy.load.html).

* La bibliothèque tierce [`joblib`](https://joblib.readthedocs.io/en/stable/index.html) permet également de sauvegarder des objets Python avec la fonction [`joblib.dump()`](https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html) et de faire l'opération inverse avec [`joblib.load()`](https://joblib.readthedocs.io/en/stable/generated/joblib.load.html), mais elle est surtout utile par rapport à `pickle` quand les objets contiennent des tableaux NumPy.

In [13]:
if not os.path.isdir(os.path.join("objets", "tennis")):
    os.makedirs(os.path.join("objets", "tennis"))

with open(os.path.join("objets", "tennis", "wta_players_2024.p"), 'wb') as fp:
    pickle.dump(creer_objets_joueuses(), fp)